# Lightweight Local SLM LoRA Fine-Tuning (16 GB MacBook Pro Edition)

Optimized for Apple Silicon Macs with 16 GB Unified Memory:
- Uses `mlx-community/Qwen2.5-1.5B-Instruct-4bit` (~900 MB footprint)
- Caps `--max-seq-length 192` with `--batch-size 2` and `--grad-checkpoint` to prevent swap
- Evaluates on a stratified 150-row holdout sample (~15 seconds) with explicit Metal cache cleanup (`mx.clear_cache()`)

In [1]:
# 1. Install MLX-LM & Dependencies into Active Kernel
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "mlx-lm", "pandas", "scikit-learn"], check=True)

CompletedProcess(args=['/Users/aloksingh/git/home-category-api/.venv-slm/bin/python', '-m', 'pip', 'install', '-q', '-U', 'mlx-lm', 'pandas', 'scikit-learn'], returncode=0)

In [2]:
# 2. Prepare & Clean Stratified ChatML Dataset
import json
import os
import re
import urllib.parse
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "main.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = Path.home() / "Downloads" / "Expense Sheet - Categorized.csv"
CORRECTIONS_PATH = PROJECT_ROOT / "data" / "corrections.csv"
LORA_DATA_DIR = PROJECT_ROOT / "lora_data"
ADAPTER_DIR = PROJECT_ROOT / "adapters" / "expense-lora-lite"
FUSED_MODEL_DIR = PROJECT_ROOT / "model" / "expense-slm-1.5b"

DIAGNOSTIC_PATTERN = (
    r'\b(?:blood\s*test|lab\s*test|diagnostic|x[\s-]*ray|mri|ct\s*scan|'
    r'ultrasound|sonography|ecg|echo|pathology|biopsy|cbc|thyroid|'
    r'lipid\s*profile|vitamin\s*[bd]|checkup|health\s*check|'
    r'thyrocare|lal\s*pathlabs|aarthi\s*scans|metropolis|orange\s*health|healthians)\b'
)

def clean_text(text: str) -> str:
    text = urllib.parse.unquote(str(text))
    text = re.sub(r'has been debited from account.*?to VPA', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'spent on your SBI Credit Card ending \d+ at', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'UPI transaction reference no.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'To check your available balance.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\d_]|[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(CSV_PATH)
df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
df = df[~df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)]
df = df[pd.to_numeric(df['amount'], errors='coerce') > 1.0].copy()

if CORRECTIONS_PATH.exists():
    corr_df = pd.read_csv(CORRECTIONS_PATH)
    corr_df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
    corr_df = corr_df[
        ~corr_df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)
    ].copy()
    corr_df['amount'] = pd.to_numeric(corr_df['amount'], errors='coerce').fillna(0.0)
    df = pd.concat([df, corr_df, corr_df], ignore_index=True)

df['Category'] = df['Category'].replace({'Shopping': 'Accessories', 'Househelp': 'House Help'})
combined_text = (df['Head'].fillna('') + ' ' + df['comment'].fillna('')).str.lower()
df.loc[combined_text.str.contains(DIAGNOSTIC_PATTERN, regex=True, na=False), 'Category'] = 'Medical'

counts = df['Category'].value_counts()
df = df[df['Category'].isin(counts[counts >= 5].index)].copy()

VALID_CATEGORIES = sorted(df['Category'].unique().tolist())
SYSTEM_PROMPT = (
    "Classify the Indian household expense into one valid category: "
    + ", ".join(VALID_CATEGORIES)
    + ". Reply with ONLY the category name."
)

def format_user_prompt(head: str, comment: str, amount: float) -> str:
    c_clean = clean_text(comment) if pd.notna(comment) and str(comment).strip() else "None"
    h_clean = str(head).strip()
    amt_str = f"₹{float(amount):.2f}" if float(amount) > 1.0 else "Unknown"
    return f"Head: {h_clean} | Comment: {c_clean} | Amount: {amt_str}"

def to_chatml_record(row) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_user_prompt(row['Head'], row['comment'], row['amount'])},
            {"role": "assistant", "content": str(row['Category']).strip()},
        ]
    }

train_df, valid_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Category'])

LORA_DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    out_path = LORA_DATA_DIR / f"{split_name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in split_df.iterrows():
            f.write(json.dumps(to_chatml_record(row), ensure_ascii=False) + "\n")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Prepared {len(train_df)} train and {len(valid_df)} validation rows across {len(VALID_CATEGORIES)} categories.")

/var/folders/v0/gz626t394gj6jhy0b_p9pjyc0000gn/T/ipykernel_58421/4155300148.py:37: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)
/var/folders/v0/gz626t394gj6jhy0b_p9pjyc0000gn/T/ipykernel_58421/4155300148.py:43: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  corr_df.rename(columns={'head': 'Head', 'category': 'Category'}, inplace=True)


Project Root: /Users/aloksingh/git/home-category-api
Prepared 6369 train and 1125 validation rows across 23 categories.


In [ ]:
# 3. Memory-Safe LoRA Fine-Tuning (1.5B 4-bit, batch_size=2, max_seq_length=192, grad_checkpoint)
import subprocess
import sys

BASE_MODEL = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

cmd = [
    sys.executable, "-m", "mlx_lm", "lora",
    "--model", BASE_MODEL,
    "--train",
    "--data", str(LORA_DATA_DIR),
    "--iters", "350",
    "--batch-size", "2",
    "--max-seq-length", "192",
    "--num-layers", "4",
    "--grad-checkpoint",
    "--learning-rate", "1.5e-4",
    "--steps-per-eval", "100",
    "--val-batches", "10",
    "--adapter-path", str(ADAPTER_DIR),
]
subprocess.run(cmd, check=True)

In [ ]:
# 4. Fast Stratified Holdout Evaluation (~15 Seconds, Low Memory)
import json
import random
from collections import defaultdict
import mlx.core as mx
from mlx_lm import load, generate
from sklearn.metrics import classification_report, accuracy_score

model, tokenizer = load(BASE_MODEL, adapter_path=str(ADAPTER_DIR))

with open(LORA_DATA_DIR / "valid.jsonl", "r", encoding="utf-8") as f:
    valid_records = [json.loads(line) for line in f]

# Stratified sample: up to 7 rows per category (~140 rows total)
random.seed(42)
by_cat = defaultdict(list)
for rec in valid_records:
    by_cat[rec["messages"][2]["content"]].append(rec)

eval_sample = []
for cat, items in by_cat.items():
    eval_sample.extend(random.sample(items, min(len(items), 7)))

print(f"Evaluating stratified sample of {len(eval_sample)} / {len(valid_records)} validation rows...")

y_true, y_pred = [], []
hallucinated = []

for idx, rec in enumerate(eval_sample, start=1):
    messages = rec["messages"][:2]
    expected = rec["messages"][2]["content"]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=6, verbose=False).strip()

    y_true.append(expected)
    y_pred.append(pred)
    if pred not in VALID_CATEGORIES:
        hallucinated.append((messages[1]["content"], expected, pred))

    if idx % 20 == 0:
        mx.clear_cache()

print(f"\nStratified Holdout Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print(f"Out-of-vocabulary predictions: {len(hallucinated)}")
print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
# 5. Edge-Case Regression Sanity Check
edge_cases = [
    ("Dishwasher", "", 459.0, "Grocery"),
    ("Dishwasher", "INR 32472 at Flipkart In on 10-10-2024", 32472.0, "Appliances"),
    ("Hotter recho", "149.00 debited to VPA hotstaronline@ybl JioHotstar", 149.0, "Bills"),
    ("ऑटो", "", 107.0, "Travel"),
    ("Detergent powder", "", 666.0, "Grocery"),
    ("Lunch", "COMPASS INDIA FOOD SERVICES PR", 630.0, "Food Outside"),
    ("Blood test", "Thyrocare", 1200.0, "Medical"),
]

for head, comment, amount, expected in edge_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_user_prompt(head, comment, amount)},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=6, verbose=False).strip()
    status = "PASS" if pred == expected else "FAIL"
    print(f"{status} {head:<20} (₹{amount:>8.2f}) -> Expected: {expected:<14} | Actual: {pred}")

# Free model from memory before fusing on disk
del model, tokenizer
mx.clear_cache()

In [ ]:
# 6. Fuse LoRA Adapter into Standalone Model Directory
import subprocess
import sys

cmd = [
    sys.executable, "-m", "mlx_lm", "fuse",
    "--model", BASE_MODEL,
    "--adapter-path", str(ADAPTER_DIR),
    "--save-path", str(FUSED_MODEL_DIR),
]
subprocess.run(cmd, check=True)
print(f"Fused standalone 1.5B SLM saved to {FUSED_MODEL_DIR}")